# FLAIR Group 4: Multimodal Mood and Theme Tagging of Music
**Late fusion of pre-trained audio features and track text metadata on MTG-Jamendo (mood/theme, split-0)**

This notebook reproduces every number reported in Sections 3–4 of the paper. Its sections follow the paper:

| Notebook section | Paper section |
|---|---|
| 1. Setup and requirements | 4.3 Protocol (reproducibility) |
| 2. Dataset and official split | 4.1 Dataset and Preprocessing |
| 3. Track text metadata | 3.2 Text branch, 4.1 |
| 4. Tag-frequency groups and evaluation functions | 4.2 Baselines and Evaluation |
| 5. Text-only baseline | 3.2 Text branch, 4.3 |
| 6. Audio branch (MusiCNN embeddings) | 3.2 Audio branch |
| 7. Audio-only baseline and late fusion | 3.3 Late fusion, 4.2 |
| 8. Success criteria and final test evaluation | 4.2, 4.3 |

**Run switches** (Section 1): `RUN_AUDIO` downloads the audio and extracts MusiCNN embeddings (long-running; see Section 6).
`RUN_FINAL_TEST` evaluates on the held-out test split and must only be switched on once, after all model choices are frozen.

## 1. Setup and requirements

In [1]:
%pip install -q scikit-learn==1.6.1 essentia-tensorflow==2.1b6.dev1389

In [2]:
import os, sys, csv, json, hashlib, platform, shutil, subprocess, tarfile, urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

RUN_AUDIO = False
RUN_FINAL_TEST = False

SEED = 42
np.random.seed(SEED)

BASE_DIR = '/content' if os.path.isdir('/content') else os.getcwd()
REPO_DIR = os.path.join(BASE_DIR, 'mtg-jamendo-dataset')
DATA_DIR = os.path.join(REPO_DIR, 'data')
SPLIT_DIR = os.path.join(DATA_DIR, 'splits', 'split-0')
OUTPUT_DIR = os.path.join(BASE_DIR, 'FLAIR_Group_4_results')
EMB_DIR = os.path.join(OUTPUT_DIR, 'musicnn_embeddings')
os.makedirs(EMB_DIR, exist_ok=True)

DATASET_COMMIT = 'cafd8e20c265ed84f1e61f1c875327971f43a62f'

print('Run switches -> RUN_AUDIO:', RUN_AUDIO, '| RUN_FINAL_TEST:', RUN_FINAL_TEST)
print('Base directory:', BASE_DIR)

Run switches -> RUN_AUDIO: False | RUN_FINAL_TEST: False
Base directory: D:\FLAIR-Group-4\run


In [3]:
versions = {
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit-learn': sklearn.__version__,
    'joblib': joblib.__version__,
}
try:
    import essentia
    versions['essentia'] = essentia.__version__
except ImportError:
    versions['essentia'] = 'not installed (only needed when RUN_AUDIO = True)'
for name, version in versions.items():
    print(f'{name:>13}: {version}')

       python: 3.12.2
        numpy: 2.5.4
       pandas: 3.1.0
 scikit-learn: 1.6.1
       joblib: 1.6.0
     essentia: not installed (only needed when RUN_AUDIO = True)


## 2. Dataset and official split
MTG-Jamendo mood/theme subset (Bogdanov et al. 2019). We clone the dataset repository at a fixed commit,
load the mood/theme annotations, and use the official **split-0** train/validation/test partitions.

In [4]:
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/MTG/mtg-jamendo-dataset.git', REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-q', DATASET_COMMIT], check=True)
head = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('MTG-Jamendo commit:', head)

MTG-Jamendo commit: cafd8e20c265ed84f1e61f1c875327971f43a62f


In [5]:
def read_tag_file(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        next(f)
        for line in f:
            if not line.strip():
                continue
            parts = line.rstrip('\n').split('\t')
            rows.append(parts[:5] + [parts[5:]])
    return pd.DataFrame(rows, columns=['TRACK_ID', 'ARTIST_ID', 'ALBUM_ID', 'PATH', 'DURATION', 'TAGS'])

df = read_tag_file(os.path.join(DATA_DIR, 'autotagging_moodtheme.tsv'))
tag_counts = Counter(tag for tags in df['TAGS'] for tag in tags)
print('Full annotation file:', df.shape[0], 'tracks,', len(tag_counts), 'unique tags')
display(df.head())

Full annotation file: 18486 tracks, 59 unique tags


,TRACK_ID,ARTIST_ID,ALBUM_ID,PATH,DURATION,TAGS
0,track_0000948,artist_000087,album_000149,48/948.mp3,212.7,[mood/theme---background]
1,track_0000950,artist_000087,album_000149,50/950.mp3,248.0,[mood/theme---background]
2,track_0000951,artist_000087,album_000149,51/951.mp3,199.7,[mood/theme---background]
3,track_0002165,artist_000326,album_000347,65/2165.mp3,229.0,[mood/theme---film]
4,track_0002263,artist_000320,album_000366,63/2263.mp3,494.7,[mood/theme---melancholic]


In [6]:
splits = {s: read_tag_file(os.path.join(SPLIT_DIR, f'autotagging_moodtheme-{s}.tsv'))
          for s in ['train', 'validation', 'test']}
for s, d in splits.items():
    print(f'{s:>10}: {len(d):>5} tracks')
print(f'{"total":>10}: {sum(len(d) for d in splits.values()):>5} tracks')

     train:  9949 tracks
validation:  3802 tracks
      test:  4231 tracks
     total: 17982 tracks


In [7]:
pairs = [('train', 'validation'), ('train', 'test'), ('validation', 'test')]
for column in ['TRACK_ID', 'ARTIST_ID', 'ALBUM_ID']:
    overlaps = {f'{a}/{b}': len(set(splits[a][column]) & set(splits[b][column])) for a, b in pairs}
    print(f'{column:>9} overlap:', overlaps)
    assert all(v == 0 for v in overlaps.values()), f'{column} overlap between partitions!'
print('No track, artist, or album appears in more than one partition.')

 TRACK_ID overlap: {'train/validation': 0, 'train/test': 0, 'validation/test': 0}


ARTIST_ID overlap: {'train/validation': 0, 'train/test': 0, 'validation/test': 0}
 ALBUM_ID overlap: {'train/validation': 0, 'train/test': 0, 'validation/test': 0}
No track, artist, or album appears in more than one partition.


In [8]:
official_tags = sorted({t for d in splits.values() for tags in d['TAGS'] for t in tags})
print('Official tags in split-0:', len(official_tags))
print('Tags only in the full file:', sorted(set(tag_counts) - set(official_tags)))

mlb = MultiLabelBinarizer(classes=official_tags)
mlb.fit([official_tags])
Y = {s: mlb.transform(d['TAGS']) for s, d in splits.items()}
for s in Y:
    print(f'{s:>10}: label matrix {Y[s].shape}, tracks without a tag: {(Y[s].sum(axis=1) == 0).sum()}')

Official tags in split-0: 56
Tags only in the full file: ['mood/theme---ambiental', 'mood/theme---horror', 'mood/theme---mellow']
     train: label matrix (9949, 56), tracks without a tag: 0
validation: label matrix (3802, 56), tracks without a tag: 0
      test: label matrix (4231, 56), tracks without a tag: 0


## 3. Track text metadata
`raw.meta.tsv` contains the track title, artist name and album name. Some album names begin with a double quote,
so the file is parsed **line by line** (no CSV quoting rules). This loads every row directly, with no recovery step.

In [9]:
def load_metadata(path):
    records, irregular = [], 0
    with open(path, encoding='utf-8') as f:
        next(f)
        for line in f:
            if not line.strip():
                continue
            parts = line.rstrip('\n').split('\t')
            text_fields = parts[3:-2]
            if len(text_fields) != 3:
                irregular += 1
                text_fields = [t for t in text_fields if t != ''] if len(text_fields) > 3 else text_fields
                if len(text_fields) > 3:
                    text_fields = text_fields[:2] + ['\t'.join(text_fields[2:])]
            records.append({'TRACK_ID': parts[0], 'TRACK_NAME': text_fields[0],
                            'ARTIST_NAME': text_fields[1], 'ALBUM_NAME': text_fields[2]})
    return pd.DataFrame(records), irregular

raw_lines = sum(1 for _ in open(os.path.join(DATA_DIR, 'raw.meta.tsv'), encoding='utf-8')) - 1
metadata, irregular = load_metadata(os.path.join(DATA_DIR, 'raw.meta.tsv'))
print('Lines in raw.meta.tsv:', raw_lines, '| rows loaded:', len(metadata), '| rows with extra tabs:', irregular)
assert len(metadata) == raw_lines and metadata['TRACK_ID'].is_unique

Lines in raw.meta.tsv: 56639 | rows loaded: 56639 | rows with extra tabs: 2


In [10]:
text_lookup = metadata.set_index('TRACK_ID')
for s, d in splits.items():
    t = text_lookup.loc[d['TRACK_ID']]
    d['TEXT_INPUT'] = (t['TRACK_NAME'] + ' ' + t['ARTIST_NAME'] + ' ' + t['ALBUM_NAME']).str.strip().values
    print(f'{s:>10}: missing metadata = {t.isna().any(axis=1).sum()}, empty text = {(d["TEXT_INPUT"] == "").sum()}')
display(splits['train'][['TRACK_ID', 'TEXT_INPUT', 'TAGS']].head())

     train: missing metadata = 0, empty text = 0
validation: missing metadata = 0, empty text = 0
      test: missing metadata = 0, empty text = 0


,TRACK_ID,TEXT_INPUT,TAGS
0,track_0000948,Victory Alexander Blu May,[mood/theme---background]
1,track_0000950,May Alexander Blu May,[mood/theme---background]
2,track_0000951,Nostalgia Alexander Blu May,[mood/theme---background]
3,track_0002165,Le nettoyeur Greg Baumont Wood,[mood/theme---film]
4,track_0002263,On the other side Sköhl Offensrüh First Act.,[mood/theme---melancholic]


## 4. Tag-frequency groups and evaluation functions
**Uncommon tags** = the bottom 25% of the 56 tags by number of positive training examples (14 tags).
Ties are broken alphabetically (stable sort over the alphabetical tag list).

Metrics (paper Section 4.2):
* **PR-AUC-macro**: average precision (area under the precision–recall curve) per tag, averaged with equal weight over tags. Primary metric, as in MediaEval.
* **PR-AUC-micro**: average precision over all track–tag decisions pooled together.
* **ROC-AUC-macro**: per-tag ROC-AUC averaged over tags (MediaEval's second official metric).
* **Uncommon-tag PR-AUC**: PR-AUC-macro over the 14 uncommon tags only.

In [11]:
train_counts = Y['train'].sum(axis=0)
order = np.argsort(train_counts, kind='stable')
N_UNCOMMON = len(official_tags) // 4
uncommon_idx = np.sort(order[:N_UNCOMMON])
common_idx = np.sort(order[N_UNCOMMON:])

short = lambda t: t.replace('mood/theme---', '')
print('Uncommon tags (training count):')
print(', '.join(f'{short(official_tags[i])} ({train_counts[i]})' for i in order[:N_UNCOMMON]))
print('Training-count cut-off: uncommon <=', train_counts[order[N_UNCOMMON - 1]],
      '| next tag has', train_counts[order[N_UNCOMMON]])

Uncommon tags (training count):
sexy (59), fast (62), groovy (78), travel (89), heavy (98), holiday (98), powerful (106), funny (109), party (127), hopeful (137), nature (137), retro (139), children (152), trailer (159)
Training-count cut-off: uncommon <= 159 | next tag has 169


In [12]:
def evaluate(y_true, probs):
    return {
        'PR-AUC-macro': average_precision_score(y_true, probs, average='macro'),
        'PR-AUC-micro': average_precision_score(y_true, probs, average='micro'),
        'ROC-AUC-macro': roc_auc_score(y_true, probs, average='macro'),
        'PR-AUC-macro (uncommon 14)': average_precision_score(y_true[:, uncommon_idx], probs[:, uncommon_idx], average='macro'),
        'PR-AUC-macro (other 42)': average_precision_score(y_true[:, common_idx], probs[:, common_idx], average='macro'),
    }

def group_pr_auc(y_true, probs, idx):
    scores = [average_precision_score(y_true[:, j], probs[:, j]) for j in idx if y_true[:, j].sum() > 0]
    return float(np.mean(scores))

def paired_bootstrap(y_true, probs_a, probs_b, idx, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    n = y_true.shape[0]
    diffs = []
    for _ in range(n_boot):
        s = rng.integers(0, n, n)
        diffs.append(group_pr_auc(y_true[s], probs_a[s], idx) - group_pr_auc(y_true[s], probs_b[s], idx))
    observed = group_pr_auc(y_true, probs_a, idx) - group_pr_auc(y_true, probs_b, idx)
    low, high = np.percentile(diffs, [2.5, 97.5])
    return observed, low, high

## 5. Text-only baseline
TF-IDF (unigrams + bigrams, max 10,000 features, min document frequency 2) fitted on **training text only**,
then one-vs-rest logistic regression (liblinear, max 1,000 iterations, random state 42).

In [13]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
X_text = {'train': tfidf.fit_transform(splits['train']['TEXT_INPUT'])}
for s in ['validation', 'test']:
    X_text[s] = tfidf.transform(splits[s]['TEXT_INPUT'])
print({s: X.shape for s, X in X_text.items()}, '| vocabulary size:', len(tfidf.vocabulary_))

text_model = OneVsRestClassifier(
    LogisticRegression(max_iter=1000, solver='liblinear', random_state=SEED), n_jobs=-1)
text_model.fit(X_text['train'], Y['train'])

P_text_val = text_model.predict_proba(X_text['validation'])
text_val_results = evaluate(Y['validation'], P_text_val)
print('\nText-only baseline, VALIDATION:')
for k, v in text_val_results.items():
    print(f'  {k:<28} {v:.4f}')

{'train': (9949, 10000), 'validation': (3802, 10000), 'test': (4231, 10000)} | vocabulary size: 10000



Text-only baseline, VALIDATION:
  PR-AUC-macro                 0.1163
  PR-AUC-micro                 0.1210
  ROC-AUC-macro                0.6281
  PR-AUC-macro (uncommon 14)   0.0991
  PR-AUC-macro (other 42)      0.1220


In [14]:
per_tag = pd.DataFrame({
    'Tag': [short(t) for t in official_tags],
    'Training count': train_counts,
    'Validation count': Y['validation'].sum(axis=0),
    'Uncommon': [i in set(uncommon_idx) for i in range(len(official_tags))],
    'Text AP': [average_precision_score(Y['validation'][:, j], P_text_val[:, j]) for j in range(len(official_tags))],
}).sort_values('Training count').reset_index(drop=True)
display(per_tag.round(4))

,Tag,Training count,Validation count,Uncommon,Text AP
0,sexy,59,37,True,0.0137
1,fast,62,26,True,0.0114
2,groovy,78,74,True,0.0343
3,travel,89,49,True,0.0195
4,holiday,98,35,True,0.3650
5,heavy,98,33,True,0.1254
6,powerful,106,47,True,0.0170
7,funny,109,23,True,0.0163
8,party,127,84,True,0.0645
9,hopeful,137,36,True,0.0111


## 6. Audio branch: MusiCNN embeddings
* **Audio:** `autotagging_moodtheme/audio-low` from the official MTG-Jamendo mirror (mono MP3, LAME VBR 2; 46 GB in 100 TAR archives, vs. 152 GB for full quality).
* **Model:** Essentia `msd-musicnn-1` (MusiCNN trained on the Million Song Dataset). Input: mono audio at 16 kHz, analysed in 3-second patches.
* **Features:** the 200-dimensional output of the penultimate layer `model/dense/BiasAdd` for each patch, **averaged over all patches of the full track** → one 200-d vector per track.

The audio is processed one TAR archive at a time (download → verify checksum → extract → embed → delete), so only ~0.5 GB of audio is on disk at once.
Embeddings are saved per archive, so the step can be resumed if Colab disconnects. Only the 17,982 split-0 tracks are embedded.

In [15]:
MODEL_PB = os.path.join(BASE_DIR, 'msd-musicnn-1.pb')
AUDIO_TYPE = 'audio-low'
MIRROR = 'https://cdn.freesound.org/mtg-jamendo/autotagging_moodtheme/' + AUDIO_TYPE + '/'
TAR_LIST = os.path.join(DATA_DIR, 'download', f'autotagging_moodtheme_{AUDIO_TYPE}_sha256_tars.txt')
AUDIO_TMP = os.path.join(BASE_DIR, 'audio_tmp')
MAX_ARCHIVES = 1

split_ids = set().union(*[set(d['TRACK_ID']) for d in splits.values()])

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

def embed_archive(tar_name, expected_sha, extractor, loader_cls):
    out_file = os.path.join(EMB_DIR, tar_name.replace('.tar', '.npz'))
    if os.path.exists(out_file):
        return 'already done'
    os.makedirs(AUDIO_TMP, exist_ok=True)
    tar_path = os.path.join(AUDIO_TMP, tar_name)
    urllib.request.urlretrieve(MIRROR + tar_name, tar_path)
    assert sha256(tar_path) == expected_sha, f'Checksum mismatch for {tar_name}'
    ids, embs = [], []
    with tarfile.open(tar_path) as tar:
        for member in tar.getmembers():
            if not member.name.endswith('.mp3'):
                continue
            track_id = 'track_' + os.path.basename(member.name).split('.')[0].zfill(7)
            if track_id not in split_ids:
                continue
            tar.extract(member, AUDIO_TMP)
            audio_path = os.path.join(AUDIO_TMP, member.name)
            try:
                audio = loader_cls(filename=audio_path, sampleRate=16000, resampleQuality=4)()
                patches = np.asarray(extractor(audio))
                ids.append(track_id)
                embs.append(patches.mean(axis=0))
            except Exception as e:
                print('   could not process', track_id, '->', e)
            os.remove(audio_path)
    emb = np.stack(embs) if embs else np.zeros((0, 200), dtype=np.float32)
    np.savez(out_file, ids=np.array(ids), emb=emb)
    os.remove(tar_path)
    return f'{len(ids)} tracks'

if RUN_AUDIO:
    import essentia.standard as es
    if not os.path.exists(MODEL_PB):
        urllib.request.urlretrieve('https://essentia.upf.edu/models/autotagging/msd/msd-musicnn-1.pb', MODEL_PB)
    extractor = es.TensorflowPredictMusiCNN(graphFilename=MODEL_PB, output='model/dense/BiasAdd')
    with open(TAR_LIST) as f:
        tar_entries = [line.split() for line in f if line.strip()]
    if MAX_ARCHIVES is not None:
        tar_entries = tar_entries[:MAX_ARCHIVES]
    for k, (expected_sha, tar_name) in enumerate(tar_entries, 1):
        print(f'[{k}/{len(tar_entries)}] {tar_name}:', embed_archive(tar_name, expected_sha, extractor, es.MonoLoader))
else:
    print('RUN_AUDIO = False -> skipping audio download and embedding extraction.')

RUN_AUDIO = False -> skipping audio download and embedding extraction.


In [16]:
emb_files = sorted(f for f in os.listdir(EMB_DIR) if f.endswith('.npz'))
if emb_files:
    emb_lookup = {}
    for f in emb_files:
        z = np.load(os.path.join(EMB_DIR, f))
        emb_lookup.update(zip(z['ids'], z['emb']))
    missing = {s: int((~d['TRACK_ID'].isin(emb_lookup.keys())).sum()) for s, d in splits.items()}
    print('Embeddings loaded:', len(emb_lookup), '| missing per partition:', missing)
    AUDIO_READY = all(v == 0 for v in missing.values())
    if AUDIO_READY:
        X_audio = {s: np.stack([emb_lookup[t] for t in d['TRACK_ID']]) for s, d in splits.items()}
        print({s: X.shape for s, X in X_audio.items()})
else:
    AUDIO_READY = False
    print('No audio embeddings yet -> audio-only and fusion results are skipped.')

No audio embeddings yet -> audio-only and fusion results are skipped.


## 7. Audio-only baseline and late fusion
The audio classifier is the same one-vs-rest logistic regression as the text branch, with the 200-d embeddings
standardised (zero mean, unit variance, fitted on training data only). Late fusion averages the two branches' probabilities:
$P_{\text{fusion}}(i,j) = \tfrac{1}{2}\,[P_{\text{audio}}(i,j) + P_{\text{text}}(i,j)]$.

In [17]:
if AUDIO_READY:
    audio_model = OneVsRestClassifier(make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver='liblinear', random_state=SEED)), n_jobs=-1)
    audio_model.fit(X_audio['train'], Y['train'])
    P_audio_val = audio_model.predict_proba(X_audio['validation'])
    P_fusion_val = (P_audio_val + P_text_val) / 2

    val_table = pd.DataFrame({
        'Audio-only': evaluate(Y['validation'], P_audio_val),
        'Text-only': text_val_results,
        'Late fusion': evaluate(Y['validation'], P_fusion_val),
    }).T
    print('VALIDATION results')
    display(val_table.round(4))
    per_tag_audio = {short(official_tags[j]): average_precision_score(Y['validation'][:, j], P_audio_val[:, j])
                     for j in range(len(official_tags))}
    per_tag['Audio AP'] = per_tag['Tag'].map(per_tag_audio)
else:
    print('Audio embeddings not available yet -> skipped.')

Audio embeddings not available yet -> skipped.


## 8. Success criteria and final test evaluation
Success criteria (paper Section 4.2), checked on validation during development and once on test at the end:
1. Our audio-only model reaches at least the official VGG-ish baseline (PR-AUC-macro 0.108, test).
2. Late fusion improves uncommon-tag PR-AUC-macro by at least **0.02** over our audio-only model.
3. The 95% paired-bootstrap confidence interval (1,000 resamples of tracks) for that improvement excludes zero.
4. Late fusion does not lower overall PR-AUC-macro relative to the audio-only model.

In [18]:
def check_success(y_true, p_audio, p_fusion, label):
    audio_res, fusion_res = evaluate(y_true, p_audio), evaluate(y_true, p_fusion)
    diff, low, high = paired_bootstrap(y_true, p_fusion, p_audio, uncommon_idx)
    checks = {
        '1. Audio-only PR-AUC-macro >= 0.108': audio_res['PR-AUC-macro'] >= 0.108,
        '2. Uncommon-tag gain >= 0.02': diff >= 0.02,
        '3. 95% CI of gain excludes 0': low > 0,
        '4. Overall PR-AUC-macro not lower': fusion_res['PR-AUC-macro'] >= audio_res['PR-AUC-macro'],
    }
    print(f'{label}: uncommon-tag gain = {diff:.4f} (95% CI {low:.4f} to {high:.4f})')
    for name, passed in checks.items():
        print(f'  {"PASS" if passed else "FAIL"}  {name}')
    return checks

if AUDIO_READY:
    check_success(Y['validation'], P_audio_val, P_fusion_val, 'VALIDATION')
else:
    print('Audio embeddings not available yet -> success criteria not evaluated.')

Audio embeddings not available yet -> success criteria not evaluated.


In [19]:
if RUN_FINAL_TEST and AUDIO_READY:
    P_text_test = text_model.predict_proba(X_text['test'])
    P_audio_test = audio_model.predict_proba(X_audio['test'])
    P_fusion_test = (P_audio_test + P_text_test) / 2
    test_table = pd.DataFrame({
        'Audio-only': evaluate(Y['test'], P_audio_test),
        'Text-only': evaluate(Y['test'], P_text_test),
        'Late fusion': evaluate(Y['test'], P_fusion_test),
    }).T
    print('TEST results')
    display(test_table.round(4))
    check_success(Y['test'], P_audio_test, P_fusion_test, 'TEST')
else:
    print('Test split not used (RUN_FINAL_TEST is False or audio not ready).')

Test split not used (RUN_FINAL_TEST is False or audio not ready).


In [20]:
joblib.dump(text_model, os.path.join(OUTPUT_DIR, 'text_model.joblib'))
joblib.dump(tfidf, os.path.join(OUTPUT_DIR, 'tfidf_vectorizer.joblib'))
joblib.dump(official_tags, os.path.join(OUTPUT_DIR, 'tag_list.joblib'))
np.save(os.path.join(OUTPUT_DIR, 'text_validation_probs.npy'), P_text_val)
np.save(os.path.join(OUTPUT_DIR, 'validation_labels.npy'), Y['validation'])
per_tag.to_csv(os.path.join(OUTPUT_DIR, 'per_tag_validation_results.csv'), index=False)
with open(os.path.join(OUTPUT_DIR, 'run_info.json'), 'w') as f:
    json.dump({'versions': versions, 'dataset_commit': DATASET_COMMIT, 'seed': SEED,
               'uncommon_tags': [short(official_tags[i]) for i in uncommon_idx],
               'text_validation': text_val_results}, f, indent=2)
if AUDIO_READY:
    joblib.dump(audio_model, os.path.join(OUTPUT_DIR, 'audio_model.joblib'))
    np.save(os.path.join(OUTPUT_DIR, 'audio_validation_probs.npy'), P_audio_val)
    val_table.to_csv(os.path.join(OUTPUT_DIR, 'validation_results.csv'))
print('Saved to', OUTPUT_DIR, ':', sorted(os.listdir(OUTPUT_DIR)))

Saved to D:\FLAIR-Group-4\run\FLAIR_Group_4_results : ['musicnn_embeddings', 'per_tag_validation_results.csv', 'run_info.json', 'tag_list.joblib', 'text_model.joblib', 'text_validation_probs.npy', 'tfidf_vectorizer.joblib', 'validation_labels.npy']
